In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from config import SimConfig
from src.data_processing import load_and_cache_entire_fleet
from src.plants.hybrid_plant import FuelCellBatteryPlant
from src.utils.evaluation import VoyageBenchmarker
from src.plotting import plot_dashboard
import numpy as np
from src.data_processing import downsample_block_mean, fit_dtmc
from src.plants.hybrid_plant import calculate_fc_cost_per_second

In [ ]:
print("Initializing MARINER Validation Environment...")

# 1. Single unified configuration
config = SimConfig() 

# 2. Load the 1 Hz SOV dataset into RAM
fleet_cache = load_and_cache_entire_fleet()

# 3. Instantiate the benchmarking engine
benchmarker = VoyageBenchmarker(fleet_cache)

# 4. The single physical truth: The Hybrid Plant
plant = FuelCellBatteryPlant(config)

In [ ]:
tolerance=2

approaches = {
    # ---------------------------------------------------------
    # 1. BASELINE STOCHASTIC DYNAMIC PROGRAMMING (H2-Only)
    # ---------------------------------------------------------
    "DP_Baseline": {
        "strategy": "SDP",
        "is_hybrid": False,   # Triggers BaselineSDPSolver & NaiveHybridWrapper
        "config": config,
        "plant": plant
    },
    
    # ---------------------------------------------------------
    # 2. EXPECTED COST HEURISTICS (H2-Only Context -> Wrapped)
    # ---------------------------------------------------------
    "ECH_Discrete_Search": {
        "strategy": "EXPECTED_COST_DISCRETE",
        "is_hybrid": False,  
        "config": config,
        "plant": plant,
        "tolerance": tolerance 
    },
    
    "ECH_Target_Step": {
        "strategy": "EXPECTED_COST_STEP",
        "is_hybrid": False,  
        "config": config,
        "plant": plant,
        "tolerance": tolerance 
    },

    # ---------------------------------------------------------
    # 3. RULE-BASED BASELINE (H2-Only Context -> Wrapped)
    # ---------------------------------------------------------
    "Threshold_Baseline": {
        "strategy": "HEURISTIC",
        "is_hybrid": False,  
        "config": config,
        "plant": plant
    }
}

In [ ]:
train_days = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14] 
test_day = 13

# Execute the simulations inside the high-fidelity HybridSimulator
df_results, sims = benchmarker.compare_approaches(approaches, train_days, test_day)


# Generate identical 4-panel dashboards for visual comparison
for approach_name, sim_instance in sims.items():
    plot_dashboard(
        sim=sim_instance, 
        approach_name=approach_name, 
        test_day=test_day, 
        layout='grid'
    )

In [ ]:
# Run Leave-One-Out for the SDP Baseline
print("\n--- APPROACH A: LEAVE ONE OUT (DP Baseline) ---")
df_loo_dp_baseline = benchmarker.run_leave_one_out(approaches["DP_Baseline"])

# Run Leave-One-Out for the new ECH Discrete Search
print("\n--- APPROACH A: LEAVE ONE OUT (ECH Discrete Search) ---")
df_loo_ech_discrete = benchmarker.run_leave_one_out(approaches["ECH_Discrete_Search"])
# Run Leave-One-Out for the new ECH Target Step
print("\n--- APPROACH A: LEAVE ONE OUT (ECH Target Step) ---")
df_loo_ech_target = benchmarker.run_leave_one_out(approaches["ECH_Target_Step"])

# Run Leave-One-Out for the Threshold Heuristic Baseline
print("\n--- APPROACH A: LEAVE ONE OUT (Threshold Baseline) ---")
df_loo_threshold = benchmarker.run_leave_one_out(approaches["Threshold_Baseline"])

In [ ]:
import pandas as pd

def print_raw_heuristic_matrices(train_days, config, fleet_cache, tolerance: int = 3):
    # 1. Prepare training data
    train_t, train_pd = [], []
    for d in train_days:
        data = fleet_cache[d]
        t_off = train_t[-1][-1] if train_t else 0.0
        train_t.append(data['t'] + t_off)
        train_pd.append(data['Pd'])
    
    t_concat = np.concatenate(train_t)
    pd_concat = np.concatenate(train_pd)
    
    # 2. Fit Macro DTMC
    ds_train_macro = downsample_block_mean(t_concat, pd_concat, config.Ts, align='t0')
    mc_macro = fit_dtmc(ds_train_macro['Pd'], config.n_states, config.alpha)
    
    p_vals = mc_macro['levels']
    n_vals = config.n_vals
    trans_mat = mc_macro.get('P', mc_macro.get('trans_mat'))
    
    # 3. Build Cost Matrix & Find Absolute Ideal Targets
    cost_matrix = np.zeros((len(n_vals), len(p_vals)))
    for i, n_val in enumerate(n_vals):
        for j, p_val in enumerate(p_vals):
            p_module = p_val / n_val
            module_cost = calculate_fc_cost_per_second(
                p_module, config.p_nom, config.k_h2, config.k_fc, 
                config.tau_fc, config.a0, config.a1, config.a2, config.alpha_deg
            )
            cost_matrix[i, j] = module_cost * n_val
            
    ideal_targets = np.argmin(cost_matrix, axis=0)
    
    # 4. Define Top-M Comfort Zones (K_zones)
    # K_zones[j] stores the demand indices 'i' where module count 'j' is in the Top M cheapest
    K_zones = {j: [] for j in range(len(n_vals))}
    for i in range(len(p_vals)):
        # Sort the column of costs for this demand state ascending
        costs_l = cost_matrix[:, i]
        top_m_indices = np.argsort(costs_l)[:tolerance]
        
        # Map this demand state to the comfort zones of those top M module counts
        for j in top_m_indices:
            K_zones[j].append(i)
            
    # 5. Calculate Expected Holding Times E[T_hold] based on Top-M zones
    T_hold = np.zeros((len(p_vals), len(n_vals)))
    for j in range(len(n_vals)):
        K_j_indices = np.array(K_zones[j], dtype=int)
        
        if len(K_j_indices) > 0:
            P_hold_j = np.sum(trans_mat[:, K_j_indices], axis=1)
        else:
            P_hold_j = np.zeros(len(p_vals))
            
        for i in range(len(p_vals)):
            p_h = P_hold_j[i]
            if p_h >= 0.9999:
                T_hold[i, j] = config.Ts * 100
            else:
                T_hold[i, j] = config.Ts / (1.0 - p_h)
                
    # --- 6. PRINT EXPLICIT MARKDOWN ENTRIES FOR THE CHAT ---
    print("\n" + "="*50)
    print("1. ABSOLUTE IDEAL TARGET MODULE BOUNDARIES")
    print("="*50)
    df_targets = pd.DataFrame({
        "Demand Level index": np.arange(len(p_vals)),
        "Demand Power [kW]": np.round(p_vals, 1),
        "Absolute Ideal Modules (n)": n_vals[ideal_targets]
    })
    print(df_targets.to_markdown(index=False))
    
    print("\n" + "="*50)
    print(f"2. EXPECTED STATE HOLDING TIMES E[T_hold] (Tolerance: Top {tolerance} Cheapest)")
    print("="*50)
    df_thold = pd.DataFrame(
        np.round(T_hold, 1), 
        index=[f"P_d={p:.1f}kW" for p in p_vals],
        columns=[f"n={n}" for n in n_vals]
    )
    print(df_thold.to_markdown())

# Run the string matrix printing engine
train_days = [4, 5, 6, 7, 8, 9, 10, 11, 12, 14]

# You can easily test different tolerances here
print_raw_heuristic_matrices(train_days, config, fleet_cache, tolerance=10)